# 🧹 Topic 17 — Data Cleaning Fundamentals with Pandas
### Week 3 · IT for Youth Ghana · Data Analytics with Python

---

**How to use this notebook**
- Run cells top to bottom — **Shift + Enter**
- **🔨 Try It** cells are yours to complete
- **⚡ Live Coding** cells — follow along with your instructor

> 📁 **Before you start:** Upload `./data/students_messy.csv` to your Colab session
**Self-study notebooks for extra reading**
- Want a slower cleaning-audit checklist? Read [Topic 17A — Data Quality Audit](topic17a_data_quality_audit.ipynb).
- Need more practice with text/category cleaning? Read [Topic 17B — Cleaning Text and Categories](topic17b_cleaning_text_categories.ipynb).
- Unsure about duplicate IDs? Read [Topic 17C — Duplicates and IDs](topic17c_duplicates_and_ids.ipynb).
- Need more numeric/missing-value practice? Read [Topic 17D — Numeric Cleaning and Missing Values](topic17d_numeric_cleaning_and_missing_values.ipynb).
- Want more date and calculated-column examples? Read [Topic 17E — Dates and Calculated Columns](topic17e_dates_and_calculated_columns.ipynb).
- Want a final audit/export pattern? Read [Topic 17F — Post-Cleaning Audit and Export](topic17f_post_cleaning_audit_export.ipynb).
- Want extra hospital dataset practice? Read [Topic 17G — Hospital Cleaning Practice](topic17g_hospital_cleaning_practice.ipynb).

**Assignment**
- Complete [Topic 17 Assignment — Hospital Patient Data Cleaning](assignments/topic17_assignment_hospital_cleaning.ipynb).


---
## 🎯 Why Data Cleaning Matters

> *"Data scientists spend up to 80% of their time cleaning and preparing data."*  
> — IBM Data Analytics Report

Raw data collected in the real world is almost never analysis-ready.
It comes from human entry, legacy systems, merged sources, and broken exports.
If you run calculations on dirty data, your results will be wrong —
and the worst part is you may not even know it.

**This topic teaches you to find every problem and fix it systematically.**

We will work through the Ghana student records dataset which has
10 different engineered data quality problems — the same kinds you will
encounter on your first day in any analytics role in Ghana.
**Extra reading:** [Topic 17A — Data Quality Audit](topic17a_data_quality_audit.ipynb) gives a slower checklist for finding problems before fixing them.


In [ ]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv("./data/students_messy.csv")

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print("First look:")
print(df.head(8))


In [ ]:
# The full picture — what are we dealing with?
print("DATASET OVERVIEW")
print("=" * 50)
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Unique values per categorical column:")
for col in ["region","school_type","year_group","gender"]:
    print(f"  {col}: {df[col].nunique()} unique — {df[col].unique()[:6]}")


---
## 🗺️ The Cleaning Plan

Before writing a single fix, we document every problem we found.
This is standard professional practice — a **data audit** before a **data fix**.

| # | Problem Found | Column(s) | Fix Strategy |
|---|--------------|-----------|-------------|
| 1 | Whitespace in names | `full_name` | `.str.strip()` |
| 2 | Duplicate student IDs | `student_id` | `.drop_duplicates()` |
| 3 | Region capitalisation | `region` | `.str.title()` + corrections dict |
| 4 | school_type variants | `school_type` | mapping dict |
| 5 | math_score as string % | `math_score` | `.str.replace()` + `.astype()` |
| 6 | Impossible scores | `math_score` | boolean mask → `NaN` |
| 7 | Missing scores | `english_score`, `science_score` | `.fillna(median)` |
| 8 | year_group variants | `year_group` | mapping dict |
| 9 | gender variants | `gender` | mapping dict |
| 10 | Mixed date formats | `date_enrolled` | `pd.to_datetime()` with `dayfirst` |

We fix one problem at a time, verify each fix, then move to the next.
**Extra reading:** The audit-to-plan workflow is expanded in [Topic 17A — Data Quality Audit](topic17a_data_quality_audit.ipynb).


---
### Fix 1 — Whitespace in Student Names
**Extra reading:** [Topic 17B — Cleaning Text and Categories](topic17b_cleaning_text_categories.ipynb) gives more examples of `.str.strip()`, case cleaning, and category mapping.


In [ ]:
# Detect the problem
names_with_space = df["full_name"].str.startswith("  ").sum()
print(f"Names with leading whitespace: {names_with_space}")
print()
print("Examples of affected names (repr shows hidden spaces):")
bad_names = df[df["full_name"].str.startswith(" ")]["full_name"].head(4)
for n in bad_names:
    print(f"  {repr(n)}")


In [ ]:
# Fix it
df["full_name"] = df["full_name"].str.strip()

# Verify
remaining = df["full_name"].str.startswith(" ").sum()
print(f"Names with leading whitespace after fix: {remaining}")
print("✅ Fix 1 complete")


---
### Fix 2 — Duplicate Student IDs
**Extra reading:** [Topic 17C — Duplicates and IDs](topic17c_duplicates_and_ids.ipynb) explains duplicate rows versus duplicate identifiers.


In [ ]:
# How many duplicates?
n_dups = df["student_id"].duplicated().sum()
print(f"Duplicate student_id rows: {n_dups}")
print()

# Show the duplicate records
dup_ids = df[df["student_id"].duplicated(keep=False)].sort_values("student_id")
print("Duplicate records:")
print(dup_ids[["student_id","full_name","region","math_score"]].to_string(index=False))


In [ ]:
# Keep first occurrence, drop the rest
df_before = len(df)
df = df.drop_duplicates(subset="student_id", keep="first")
df_after = len(df)

print(f"Rows before: {df_before}")
print(f"Rows after:  {df_after}")
print(f"Removed:     {df_before - df_after} duplicate rows")
print("✅ Fix 2 complete")


---
### Fix 3 — Region Capitalisation & Abbreviations


In [ ]:
print("Raw region values:")
print(df["region"].value_counts().to_string())


In [ ]:
# Step 1: title case handles most variants
df["region"] = df["region"].str.strip().str.title()

# Step 2: fix remaining abbreviations
corrections = {
    "Gr. Accra":       "Greater Accra",
    "G.Accra":         "Greater Accra",
    "Asante":          "Ashanti",
    "Western Region":  "Western",
    "Eastern Region":  "Eastern",
    "U/E":             "Upper East",
}
df["region"] = df["region"].replace(corrections)

print("Cleaned region values:")
print(df["region"].value_counts().to_string())
print()
print(f"Unique regions: {df['region'].nunique()}")
print("✅ Fix 3 complete")


---
### Fix 4 — School Type Variants


In [ ]:
print("Raw school_type values:")
print(df["school_type"].value_counts().to_string())


In [ ]:
# Map all variants to "Public" or "Private"
def standardise_school_type(val):
    if pd.isna(val):
        return val
    v = str(val).strip().lower()
    public_set  = {"public","govt","government","gov't"}
    private_set = {"private","pvt","priv"}
    if v in public_set:
        return "Public"
    elif v in private_set:
        return "Private"
    return "Unknown"

df["school_type"] = df["school_type"].apply(standardise_school_type)

print("Cleaned school_type:")
print(df["school_type"].value_counts())
print("✅ Fix 4 complete")


---
### Fix 5 & 6 — Math Score: String Format & Impossible Values
**Extra reading:** [Topic 17D — Numeric Cleaning and Missing Values](topic17d_numeric_cleaning_and_missing_values.ipynb) expands symbol removal, numeric conversion, impossible values, and median filling.


In [ ]:
print("math_score dtype:", df["math_score"].dtype)
print()
print("Sample values (first 15):")
print(df["math_score"].head(15).to_string())
print()
# Check for % sign in values
pct_count = df["math_score"].astype(str).str.contains("%").sum()
print(f"Scores stored with '%' suffix: {pct_count}")


In [ ]:
# Step 1: Strip the % sign and convert to float
df["math_score"] = df["math_score"].astype(str).str.replace("%","",regex=False)

# Step 2: Convert to numeric — coerce any non-numeric to NaN
df["math_score"] = pd.to_numeric(df["math_score"], errors="coerce")

# Step 3: Flag and remove impossible values (outside 0–100)
impossible_mask = (df["math_score"] < 0) | (df["math_score"] > 100)
impossible_count = impossible_mask.sum()
print(f"Impossible math scores found: {impossible_count}")
print("Values:", df.loc[impossible_mask, "math_score"].tolist())

# Replace impossible values with NaN
df.loc[impossible_mask, "math_score"] = np.nan

print()
print(f"math_score dtype after fix: {df['math_score'].dtype}")
print(f"Missing math scores now   : {df['math_score'].isna().sum()}")
print("✅ Fix 5 & 6 complete")


---
### Fix 7 — Missing Score Values

When scores are missing, we have two main strategies:
- **Drop rows** — only if missing data is a small fraction and random
- **Fill (impute)** — replace with mean, median, or a calculated value

For student scores we will fill with the **median per year group** —
because JHS 3 students generally score differently from JHS 1.
Using the overall median would be less accurate.


In [ ]:
# Check missing counts before
missing_before = df[["math_score","english_score","science_score"]].isnull().sum()
print("Missing before fill:")
print(missing_before)
print()

# Fill each score with the median for that year_group
for score_col in ["math_score","english_score","science_score"]:
    df[score_col] = df.groupby("year_group")[score_col]                       .transform(lambda x: x.fillna(x.median()))

# Verify
missing_after = df[["math_score","english_score","science_score"]].isnull().sum()
print("Missing after fill:")
print(missing_after)
print("✅ Fix 7 complete")


**🔨 Try It 1** — Before and after filling, how did the median score
for `english_score` change? Why might filling with the median have
less impact on the median than on the mean?

Write your reasoning as a markdown comment, then verify with code.


In [ ]:
# 🔨 TRY IT 1

# Reload fresh and calculate median before fill
df_check = pd.read_csv("./data/students_messy.csv")
median_before = df_check["english_score"].median()

# After your fix:
median_after = df["english_score"].median()

print(f"English score median BEFORE fill: {median_before}")
print(f"English score median AFTER fill : {median_after}")
print()
# YOUR EXPLANATION HERE as a comment:
# Reason the median changes less than the mean when filling missing values:
# ...


---
### Fix 8 — Year Group Variants


In [ ]:
print("Raw year_group values:")
print(df["year_group"].value_counts().to_string())


In [ ]:
year_group_map = {
    # JHS 1 variants
    "JHS1": "JHS 1", "Year 1": "JHS 1", "jhs1": "JHS 1",
    "jhs 1": "JHS 1", "Form 1": "JHS 1",
    # JHS 2 variants
    "JHS2": "JHS 2", "Year 2": "JHS 2", "jhs2": "JHS 2",
    "jhs 2": "JHS 2", "Form 2": "JHS 2",
    # JHS 3 variants
    "JHS3": "JHS 3", "Year 3": "JHS 3", "jhs3": "JHS 3",
    "jhs 3": "JHS 3", "Form 3": "JHS 3",
}

df["year_group"] = df["year_group"].replace(year_group_map)

print("Cleaned year_group:")
print(df["year_group"].value_counts().sort_index())
print("✅ Fix 8 complete")


---
### Fix 9 — Gender Variants


In [ ]:
print("Raw gender values:")
print(df["gender"].value_counts().to_string())


In [ ]:
def standardise_gender(val):
    if pd.isna(val):
        return val
    v = str(val).strip().lower()
    if v in {"male","m"}:
        return "Male"
    elif v in {"female","f"}:
        return "Female"
    return "Unknown"

df["gender"] = df["gender"].apply(standardise_gender)

print("Cleaned gender:")
print(df["gender"].value_counts())
print("✅ Fix 9 complete")


---
### Fix 10 — Mixed Date Formats
**Extra reading:** [Topic 17E — Dates and Calculated Columns](topic17e_dates_and_calculated_columns.ipynb) explains `pd.to_datetime()`, `NaT`, date parts, and new columns.


In [ ]:
# Detect the problem
sample_dates = df["date_enrolled"].head(20)
print("Sample date values:")
print(sample_dates.to_string())
print()
slash_dates = df["date_enrolled"].str.contains("/", na=False).sum()
print(f"Dates with DD/MM/YYYY format (containing '/'): {slash_dates}")


In [ ]:
# pd.to_datetime with dayfirst=True handles both formats correctly
df["date_enrolled"] = pd.to_datetime(df["date_enrolled"], dayfirst=True, errors="coerce")

print(f"date_enrolled dtype: {df['date_enrolled'].dtype}")
print()
print("Sample parsed dates:")
print(df["date_enrolled"].head(8).to_string())
print()
print(f"Unparseable dates (NaT): {df['date_enrolled'].isna().sum()}")
print("✅ Fix 10 complete")


---
## ➕ Adding Calculated Columns

After cleaning, it is common to **enrich** the dataset
by computing new columns from existing clean ones.


In [ ]:
# Average score across all three subjects
df["avg_score"] = df[["math_score","english_score","science_score"]].mean(axis=1).round(1)

# Pass/Fail flag (pass = avg score ≥ 50)
df["result"] = df["avg_score"].apply(lambda x: "Pass" if x >= 50 else "Fail")

# Years since enrolment (how long have they been at school?)
reference_date = pd.Timestamp("2024-01-01")
df["years_enrolled"] = ((reference_date - df["date_enrolled"]).dt.days / 365).round(1)

print("New columns added:")
print(df[["full_name","avg_score","result","years_enrolled"]].head(10).to_string(index=False))


**🔨 Try It 2** — Using the cleaned DataFrame:

1. How many students passed (avg_score ≥ 50)?
2. What is the pass rate per region? Which region has the highest?
3. Which year_group has the highest average math_score?
4. Is there a difference in average score between male and female students?

Write code for each question. Print clean, formatted answers.


In [ ]:
# 🔨 TRY IT 2

# 1. Total students who passed
# YOUR CODE HERE

# 2. Pass rate per region
# YOUR CODE HERE

# 3. Avg math_score per year_group
# YOUR CODE HERE

# 4. Avg score by gender
# YOUR CODE HERE


---
## 📋 Post-Cleaning Audit

Always run a final audit after cleaning to confirm every
problem has been resolved before you begin analysis.
**Extra reading:** [Topic 17F — Post-Cleaning Audit and Export](topic17f_post_cleaning_audit_export.ipynb) shows how to confirm and save clean data.


In [ ]:
print("=" * 60)
print("  POST-CLEANING DATA AUDIT")
print("=" * 60)
print()
print(f"  Rows remaining          : {len(df)}")
print(f"  Columns                 : {len(df.columns)}")
print()
print("  Missing values per column:")
missing = df.isnull().sum()
for col, count in missing.items():
    status = "✅" if count == 0 else "⚠️"
    print(f"    {status}  {col:<20} : {count}")
print()
print("  Unique values (categorical):")
for col in ["region","school_type","year_group","gender","result"]:
    vals = sorted(df[col].dropna().unique())
    print(f"    {col:<20} : {vals}")
print()
print(f"  date_enrolled dtype     : {df['date_enrolled'].dtype}")
print(f"  math_score dtype        : {df['math_score'].dtype}")
print(f"  avg_score range         : {df['avg_score'].min():.1f} – {df['avg_score'].max():.1f}")
print()
print("=" * 60)
print("  Dataset is ready for analysis ✅")
print("=" * 60)


In [ ]:
# Save the clean dataset for use in analysis
df.to_csv("students_clean.csv", index=False)
print("✅ Clean dataset saved: students_clean.csv")
print(f"   Rows: {len(df)}  |  Columns: {len(df.columns)}")


---
## ⚡ Live Coding — Hospital Patient Records
 
> The instructor cleans the students dataset live — you apply  
> **the same systematic approach** to the hospital patient data.

The problems are different in detail but identical in category —
you already know how to fix every one of them.
**Extra reading:** [Topic 17G — Hospital Cleaning Practice](topic17g_hospital_cleaning_practice.ipynb) gives extra practice on the live-coding dataset.


In [ ]:
# ⚡ LIVE CODING 1 — Load and audit

hosp = pd.read_csv("./data/hospital_patients_messy.csv")

# YOUR CODE: run the full audit
# print shape, head, info, isnull().sum()
# check unique values for categorical columns


In [ ]:
# ⚡ LIVE CODING 2 — Write your cleaning plan
# Before fixing anything, document what you found:

problems = {
    # "column": "problem description"
    # e.g. "patient_name": "trailing whitespace in ~6% of rows",
}

# YOUR CODE: fill in the problems dictionary
# then print it as a plan


In [ ]:
# ⚡ LIVE CODING 3 — Fix whitespace in patient_name
# YOUR CODE HERE


In [ ]:
# ⚡ LIVE CODING 4 — Remove duplicate patient_ids
# YOUR CODE HERE


In [ ]:
# ⚡ LIVE CODING 5 — Standardise region
# YOUR CODE HERE


In [ ]:
# ⚡ LIVE CODING 6 — Fix age column
# Some ages are float (45.0) — convert to int
# Some ages are impossible (negative or >120) — replace with NaN
# YOUR CODE HERE


In [ ]:
# ⚡ LIVE CODING 7 — Standardise gender and ward
# YOUR CODE HERE


In [ ]:
# ⚡ LIVE CODING 8 — Parse mixed date formats
# YOUR CODE HERE


In [ ]:
# ⚡ LIVE CODING 9 — Flag length_of_stay mismatches
# Rows where discharge - admission ≠ length_of_stay

# Hint: calculate the expected LOS from dates and compare
# expected_los = (hosp["discharge_date"] - hosp["admission_date"]).dt.days
# mismatch = hosp[hosp["length_of_stay"] != expected_los]
# YOUR CODE HERE


In [ ]:
# ⚡ LIVE CODING 10 — Post-cleaning audit + export
# YOUR CODE HERE


---
## ✅ What You Covered in This Notebook

| Problem Type | pandas Tool Used |
|-------------|-----------------|
| Whitespace | `.str.strip()` |
| Duplicates | `.duplicated()` `.drop_duplicates()` |
| Capitalisation | `.str.title()` `.replace()` custom mapping |
| String-encoded numbers | `.str.replace()` `pd.to_numeric(errors='coerce')` |
| Impossible values | Boolean mask → `np.nan` |
| Missing values | `.isnull()` `.fillna()` `.groupby().transform()` |
| Type conversion | `.astype()` `pd.to_datetime()` |
| Adding columns | `df["new"] = expression` `.apply()` |
| Post-audit | `.isnull().sum()` `.nunique()` `.value_counts()` |
| Export clean data | `.to_csv(index=False)` |

---

**Next up → Topic 18: Basic Data Analysis & Grouping 📊**  
Your dataset is now clean and enriched. In Topic 18 you'll use  
`.groupby()`, `.agg()`, `pd.crosstab()`, and sorting  
to produce real analytical answers from real clean data.
### Useful Online Resources

- [pandas user guide: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas user guide: Working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas user guide: Duplicate Labels](https://pandas.pydata.org/docs/user_guide/duplicates.html)
- [pandas API: `to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)
- [pandas API: `to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas API: `DataFrame.drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)
